In [7]:
"""
gnn_bb_4_runs_combined_FINAL_NO_NODE_FILTER.py

FINAL STATIC-GNN RUNNER for the reviewer-extension B&B relaxation-selection study.

There are exactly FOUR GNN experiments:

    1) Combined  -> Combined   Classification
    2) Combined  -> Combined   Regression
    3) Synthetic'-> Combined   Classification
    4) Synthetic'-> Combined   Regression

IMPORTANT
=========
There are NO Dynamic-10 / Dynamic-20 GNN experiments in this script.

The GNN input is ONLY the static tripartite QCQP graph built from the QCQP
instance (A_list, b_list, c_list, lower/upper bounds, variable/quadratic-term/
constraint nodes and their edges).

The target CSV filename contains "probe10" only because Method-2 target files
were generated under that naming convention. The GNN does NOT use probe
features. The probe10/probe20 targets are expected to be identical; this script
can audit that when the probe20 target files are present.

FINAL benchmark rules
=====================
- Combined clean target set before modeling exclusions: 283.
- Remove ONLY:
      QPLIB||QPLIB_2512
      QPLIB||QPLIB_3402
      QPLIB||QPLIB_3703
  because their zero LP-root objective makes the thesis-defined root-relative
  dynamic normalization numerically explosive elsewhere in the experiment.
  We keep the SAME eligible benchmark for every model family.
- Final Combined size MUST be 280.
- Synthetic' MUST contain 6069 instances, matching the finalized sklearn run.
- NO node-count filter.
- NO sdp_node_count < lp_node_count filter.
- NO requested-vs-actual budget filter.

Combined -> Combined split
==========================
The exact final sklearn split manifest is reused:

    .../sklearn_all_runs_synthetic_prime_8016_20261021_014902_034048/
        combined_train_test_split_manifest.csv

Thus sklearn and GNN are evaluated on literally the same Combined train/test
instances.

GNN architecture / training
===========================
Kept from the supplied gnn_bb_multi.py:
- permutation-invariant tripartite QCQP graph
- USE_FEATURE_NORM = True (train-only z-score; already implemented in the supplied GNN)
- USE_SIZE_IN_GLOBAL = False
- hidden size = 64
- 3 message-passing layers
- dropout = 0.01
- learning rate = 1e-3
- weight decay = 1e-5
- epochs = 10
- train/eval batch sizes = 8 / 16
- classification auxiliary delta loss
- regression decision auxiliary loss
- same sample weighting settings

No DI feature table and no probe-feature table is used as GNN input.
"""

from __future__ import annotations

import pickle
import shutil
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool
from torch_geometric.utils import scatter

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    mean_absolute_error,
    mean_squared_error,
)


# ============================================================
# PATHS / FINAL EXPERIMENT CONFIG
# ============================================================

ROOT = Path(r"C:\Users\suuser\Desktop\minlplib")
FINAL_DIR = ROOT / "Areview" / "combined_minlplib_qplib"
DIM_TAG = "combined_minlplib_qplib"
SYNTHETIC_FINAL_DIR = (
    ROOT / "paperexperiments" / "synthetic_prime_minimal_revision"
    / "bb_separate_trees" / "synthetic_prime_8016_fixed100_v1"
    / "postrun_clean"
)

SKLEARN_RUN_NAME = "sklearn_all_runs_synthetic_prime_8016_20261021_014902_034048"


# Static GNN uses one copy of the Method-2 targets.
TARGET_PROBE = 10

TARGET_COL = "delta"
CLASS_COL = "y"
TAU_SIGN = 0.01

EXPECTED_COMBINED_N = 280
EXPECTED_SYNTHETIC_N = 6069

EXCLUDE_COMBINED_KEYS = {
    "QPLIB||QPLIB_2512",
    "QPLIB||QPLIB_3402",
    "QPLIB||QPLIB_3703",
}
ZERO_ROOT_REASON = "lp_root_obj_zero_dynamic_normalization"

# Exact final sklearn Combined split.
SKLEARN_RESULT_ROOT = FINAL_DIR / SKLEARN_RUN_NAME
COMBINED_SPLIT_FILE = (
    SKLEARN_RESULT_ROOT / "combined_train_test_split_manifest.csv"
)

# Actual QCQP PKL directories from the final Combined feature-extraction setup.
COMBINED_PKL_DIRS: List[Tuple[str, Path]] = [
    ("MINLPLIB", ROOT / "qcp_lpall"),
    ("MINLPLIB", ROOT / "qcp_lpopf"),
    ("MINLPLIB", ROOT / "qcp_lpbigpkl"),
    (
        "QPLIB",
        ROOT
        / "Areview"
        / "qplib"
        / "pickles_nlt300_mge1_bounded",
    ),
]

# Synthetic' QCQP instances from the supplied GNN code.
SYNTHETIC_PKL = (
    ROOT / "paperexperiments" / "synthetic_prime_minimal_revision" / "instances_mixed_8016_candidate.pkl"
)

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_ROOT = FINAL_DIR / f"gnn_synprime_8016_{RUN_STAMP}"
DELETE_PREVIOUS_OUTPUT = False

MASTER_RESULTS_FILE = OUTPUT_ROOT / "all_gnn_results.csv"
THESIS_TABLE_FILE = OUTPUT_ROOT / "thesis_style_gnn_results.csv"
EXPERIMENT_MANIFEST_FILE = OUTPUT_ROOT / "experiment_manifest.csv"


def combined_target_path(probe_nodes: int) -> Path:
    return (
        FINAL_DIR
        / "targets"
        / "method2"
        / DIM_TAG
        / (
            f"targets_method2_relaxation_{DIM_TAG}"
            f"_probe{int(probe_nodes)}.csv"
        )
    )


def synthetic_target_path(probe_nodes: int) -> Path:
    return (SYNTHETIC_FINAL_DIR / "targets" / "method2" / "synthetic_prime"
            / f"targets_method2_relaxation_synthetic_prime_probe{int(probe_nodes)}.csv")


# ============================================================
# GNN HYPERPARAMETERS — PRESERVED FROM SUPPLIED CODE
# ============================================================

PERMUTATION_INVARIANT = True

USE_FEATURE_NORM = True
USE_SIZE_IN_GLOBAL = False

HIDDEN = 64
LAYERS = 3
DROPOUT = 0.01
LR = 1e-3
WD = 1e-5
EPOCHS = 10

BATCH_TRAIN = 8
BATCH_EVAL = 16

USE_SAMPLE_WEIGHTS = True
WEIGHT_ALPHA = 0.30

LAMBDA_DELTA = 0.25
USE_DECISION_AUX = True
DECISION_LOSS_WEIGHT = 0.50
DECISION_SCALE = 20.0

TOL = 1e-12
MAX_QUAD_NODES = -1
MAX_LIN_EDGES = -1
_CLIP = 1e4


# ============================================================
# GENERAL HELPERS
# ============================================================

def _ts() -> str:
    return datetime.now().strftime("%H:%M:%S")


def read_any(path: Path) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Required file does not exist:\n{path}"
        )

    suf = path.suffix.lower()

    if suf == ".csv":
        return pd.read_csv(path)

    if suf in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    if suf == ".parquet":
        return pd.read_parquet(path)

    raise ValueError(f"Unsupported table type: {path}")


def _canon_name(value) -> str:
    s = str(value).strip()
    if s.lower().endswith(".pkl"):
        s = s[:-4]
    return s


def _norm_source(value) -> str:
    return str(value).strip().upper()


def _require_columns(
    df: pd.DataFrame,
    cols: Iterable[str],
    label: str,
) -> None:
    missing = [c for c in cols if c not in df.columns]

    if missing:
        raise ValueError(
            f"{label} missing columns: {missing}\n"
            f"Available: {list(df.columns)}"
        )


def _assert_unique(
    df: pd.DataFrame,
    key: str,
    label: str,
) -> None:
    if df[key].duplicated().any():
        examples = (
            df.loc[
                df[key].duplicated(keep=False),
                key,
            ]
            .astype(str)
            .head(20)
            .tolist()
        )

        raise ValueError(
            f"{label} has duplicate {key}. Examples: {examples}"
        )


# ============================================================
# FINAL KEY HANDLING
# ============================================================

def canonicalize_combined_key(
    df: pd.DataFrame,
    label: str,
) -> pd.DataFrame:
    """
    Canonical Combined key:
        SOURCE||instance_name
    """
    out = df.copy()

    if "instance_key" in out.columns:
        out["_key"] = (
            out["instance_key"]
            .astype(str)
            .str.strip()
        )
    else:
        if "instance_name" not in out.columns:
            raise ValueError(
                f"{label}: need instance_key or instance_name."
            )

        out["instance_name"] = (
            out["instance_name"]
            .map(_canon_name)
        )

        if "source" in out.columns:
            src = out["source"].map(_norm_source)

        elif "id" in out.columns:
            ids = out["id"].astype(str).str.strip()

            src = pd.Series(
                np.where(
                    ids.str.startswith("MINLPLIB__"),
                    "MINLPLIB",
                    np.where(
                        ids.str.startswith("QPLIB__"),
                        "QPLIB",
                        "",
                    ),
                ),
                index=out.index,
                dtype=object,
            )

            if (src == "").any():
                bad = (
                    out.loc[src == "", "id"]
                    .head(20)
                    .tolist()
                )

                raise ValueError(
                    f"{label}: cannot infer source from id. "
                    f"Examples: {bad}"
                )
        else:
            raise ValueError(
                f"{label}: cannot construct Combined key."
            )

        out["_key"] = (
            src.astype(str)
            + "||"
            + out["instance_name"].astype(str)
        )

    out["_key"] = (
        out["_key"]
        .astype(str)
        .str.strip()
    )

    if "source" not in out.columns:
        out["source"] = (
            out["_key"]
            .str.split("||", n=1)
            .str[0]
        )

    out["source"] = out["source"].map(_norm_source)

    if "instance_name" not in out.columns:
        out["instance_name"] = (
            out["_key"]
            .str.split("||", n=1)
            .str[1]
        )

    out["instance_name"] = (
        out["instance_name"]
        .map(_canon_name)
    )

    _assert_unique(out, "_key", label)
    return out


def canonicalize_synthetic_key(
    df: pd.DataFrame,
    label: str,
) -> pd.DataFrame:
    """Match finalized Method-2 convention SYNTHETIC||original numeric ID."""
    out = df.copy()
    if "instance_key" in out.columns:
        raw = out["instance_key"].astype("string").str.strip()
    elif "dataset_id" in out.columns:
        raw = out["dataset_id"].astype("string").str.strip()
    elif "instance_id" in out.columns:
        raw = out["instance_id"].astype("string").str.strip()
    elif "id" in out.columns:
        raw = out["id"].astype("string").str.strip()
    else:
        raise ValueError(f"{label}: no authoritative instance key/id")
    if raw.isna().any() or raw.eq("").any():
        raise ValueError(f"{label}: blank Synthetic' identifiers")
    raw = raw.str.replace(r"^SYNTHETIC\|\|", "", regex=True)
    if not raw.str.fullmatch(r"[0-9]+").all():
        raise ValueError(f"{label}: Synthetic' keys must have numeric original IDs")
    out["_key"] = "SYNTHETIC||" + raw.astype(str)
    _assert_unique(out, "_key", label)
    return out


# ============================================================
# FINAL TARGET LOADING
# ============================================================

def apply_zero_root_exclusion(
    df: pd.DataFrame,
    dataset_kind: str,
    label: str,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if dataset_kind != "combined":
        return (
            df.reset_index(drop=True),
            pd.DataFrame(
                columns=[
                    *df.columns,
                    "remove_reason",
                ]
            ),
        )

    mask = (
        df["_key"]
        .astype(str)
        .str.strip()
        .isin(EXCLUDE_COMBINED_KEYS)
    )

    removed = df.loc[mask].copy()

    if len(removed):
        removed["remove_reason"] = ZERO_ROOT_REASON

    kept = (
        df.loc[~mask]
        .copy()
        .reset_index(drop=True)
    )

    print(
        f"[{_ts()}] [{label}] zero-LP-root exclusion: "
        f"removed {len(removed)}/{len(df)}; kept {len(kept)}",
        flush=True,
    )

    return (
        kept,
        removed.reset_index(drop=True),
    )


def load_targets(
    dataset_kind: str,
    probe_nodes: int = TARGET_PROBE,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    p = int(probe_nodes)

    if dataset_kind == "combined":
        df = canonicalize_combined_key(
            read_any(
                combined_target_path(p)
            ),
            label=f"Combined targets probe{p}",
        )

    elif dataset_kind == "synthetic":
        df = canonicalize_synthetic_key(
            read_any(
                synthetic_target_path(p)
            ),
            label=f"Synthetic' targets probe{p}",
        )

        if "source" not in df.columns:
            df["source"] = "SYNTHETIC_PRIME"

    else:
        raise ValueError(dataset_kind)

    _require_columns(
        df,
        [
            "_key",
            TARGET_COL,
            CLASS_COL,
        ],
        f"{dataset_kind} targets probe{p}",
    )

    df[TARGET_COL] = pd.to_numeric(
        df[TARGET_COL],
        errors="coerce",
    )

    df[CLASS_COL] = pd.to_numeric(
        df[CLASS_COL],
        errors="coerce",
    )

    if not np.isfinite(
        df[TARGET_COL].to_numpy(float)
    ).all():
        raise ValueError(
            f"{dataset_kind} probe{p}: non-finite delta."
        )

    bad_y = (
        df[CLASS_COL].isna()
        | ~df[CLASS_COL].isin([0, 1])
    )

    if bad_y.any():
        raise ValueError(
            f"{dataset_kind} probe{p}: invalid y."
        )

    df[CLASS_COL] = (
        df[CLASS_COL]
        .astype(int)
    )

    df, removed = apply_zero_root_exclusion(
        df,
        dataset_kind=dataset_kind,
        label=f"{dataset_kind} probe{p}",
    )

    expected = (
        EXPECTED_COMBINED_N
        if dataset_kind == "combined"
        else EXPECTED_SYNTHETIC_N
    )

    if len(df) != expected:
        raise RuntimeError(
            f"{dataset_kind} probe{p}: expected {expected} final rows, "
            f"found {len(df)}."
        )

    _assert_unique(
        df,
        "_key",
        f"{dataset_kind} final targets probe{p}",
    )

    return (
        df.reset_index(drop=True),
        removed,
    )


def audit_probe10_probe20_targets(
    dataset_kind: str,
    probe10: pd.DataFrame,
) -> None:
    """
    User clarified that Method-2 target values are the same for probe10/probe20.
    If the probe20 target file exists, audit that statement and fail if not.
    """
    path20 = (
        combined_target_path(20)
        if dataset_kind == "combined"
        else synthetic_target_path(20)
    )

    if not path20.exists():
        print(
            f"[{_ts()}] [{dataset_kind}] probe20 target file not present; "
            "skipping optional target-consistency audit.",
            flush=True,
        )
        return

    probe20, _ = load_targets(
        dataset_kind,
        probe_nodes=20,
    )

    a = (
        probe10[
            [
                "_key",
                TARGET_COL,
                CLASS_COL,
            ]
        ]
        .sort_values("_key")
        .reset_index(drop=True)
    )

    b = (
        probe20[
            [
                "_key",
                TARGET_COL,
                CLASS_COL,
            ]
        ]
        .sort_values("_key")
        .reset_index(drop=True)
    )

    if set(a["_key"]) != set(b["_key"]):
        raise RuntimeError(
            f"{dataset_kind}: probe10/probe20 target key sets differ."
        )

    if not np.array_equal(
        a[CLASS_COL].to_numpy(int),
        b[CLASS_COL].to_numpy(int),
    ):
        raise RuntimeError(
            f"{dataset_kind}: probe10/probe20 y values differ."
        )

    if not np.allclose(
        a[TARGET_COL].to_numpy(float),
        b[TARGET_COL].to_numpy(float),
        rtol=0.0,
        atol=1e-12,
        equal_nan=False,
    ):
        diff = np.max(
            np.abs(
                a[TARGET_COL].to_numpy(float)
                - b[TARGET_COL].to_numpy(float)
            )
        )

        raise RuntimeError(
            f"{dataset_kind}: probe10/probe20 delta differs. "
            f"max abs diff={diff}"
        )

    print(
        f"[{_ts()}] [{dataset_kind}] probe10/probe20 target consistency PASSED.",
        flush=True,
    )


# ============================================================
# PKL / GRAPH INSTANCE LOADING
# ============================================================

class RobustUnpickler(pickle.Unpickler):
    _cache = {}

    def find_class(self, module, name):
        try:
            return super().find_class(module, name)
        except Exception:
            key = (module, name)

            if key not in self._cache:
                self._cache[key] = type(
                    name,
                    (object,),
                    {},
                )

            return self._cache[key]


def load_pkl(path: Path) -> List[Any]:
    path = Path(path)

    with open(path, "rb") as f:
        data = RobustUnpickler(f).load()

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        # A QCQP instance dict itself.
        if "A_list" in data:
            return [data]

        if "instances" in data:
            obj = data["instances"]
            return (
                obj
                if isinstance(obj, list)
                else [obj]
            )

        return list(data.values())

    try:
        return list(data)
    except TypeError:
        return [data]


def get_field(
    obj,
    name,
    default=None,
):
    if isinstance(obj, dict):
        return obj.get(
            name,
            default,
        )

    return getattr(
        obj,
        name,
        default,
    )


def load_combined_instances(
    eligible_keys: set,
) -> Dict[Tuple[str, str], Any]:
    """
    Map:
        ("combined", "SOURCE||instance_name") -> QCQP instance
    """
    out: Dict[
        Tuple[str, str],
        Any,
    ] = {}

    for source, folder in COMBINED_PKL_DIRS:
        source = _norm_source(source)
        folder = Path(folder)

        if not folder.exists():
            raise FileNotFoundError(
                f"Combined PKL directory does not exist:\n{folder}"
            )

        files = sorted(
            folder.glob("*.pkl")
        )

        if not files:
            raise FileNotFoundError(
                f"No .pkl files found in:\n{folder}"
            )

        matched_here = 0

        for p in files:
            key = (
                f"{source}||"
                f"{_canon_name(p.stem)}"
            )

            if key not in eligible_keys:
                continue

            payload = load_pkl(p)

            if len(payload) != 1:
                raise RuntimeError(
                    f"Combined target key is file-level ({key}), "
                    f"but {p} contains {len(payload)} instances. "
                    "Cannot choose one silently."
                )

            map_key = (
                "combined",
                key,
            )

            if map_key in out:
                raise RuntimeError(
                    "Duplicate Combined QCQP key across PKL directories: "
                    f"{key}"
                )

            out[map_key] = payload[0]
            matched_here += 1

        print(
            f"[{_ts()}] [PKL] {source} | {folder.name}: "
            f"matched {matched_here} eligible target instances",
            flush=True,
        )

    found = {
        key
        for kind, key in out.keys()
        if kind == "combined"
    }

    missing = (
        eligible_keys
        - found
    )

    if missing:
        raise RuntimeError(
            "Combined QCQP PKLs missing eligible target instances. "
            f"Examples: {sorted(missing)[:30]}"
        )

    if len(found) != EXPECTED_COMBINED_N:
        raise RuntimeError(
            f"Combined graph coverage expected {EXPECTED_COMBINED_N}, "
            f"found {len(found)}."
        )

    return out


def load_synthetic_instances(eligible_keys: set) -> Dict[Tuple[str, str], Any]:
    """Use ORIGINAL Synthetic' QCQP IDs, never pickle list positions."""
    if not SYNTHETIC_PKL.is_file():
        raise FileNotFoundError(f"Synthetic' PKL does not exist: {SYNTHETIC_PKL}")
    payload = load_pkl(SYNTHETIC_PKL)
    wanted = {k.split("||", 1)[1]: k for k in eligible_keys}
    if len(wanted) != len(eligible_keys):
        raise ValueError("Synthetic keys are not unique numeric original IDs")

    # Dataset source objects must carry their original ID. If not, refuse
    # to map by order: filtered output IDs are not guaranteed to be positional.
    candidates = ("instance_id", "dataset_id", "id", "instance_index")
    best_field, best_map, best_overlap = None, None, -1
    for field in candidates:
        mapping = {}
        bad = False
        for inst in payload:
            v = get_field(inst, field, None)
            if v is None:
                continue
            raw = str(v).strip()
            if raw.startswith("SYNTHETIC||"):
                raw = raw.split("||", 1)[1]
            if raw in mapping:
                bad = True
                break
            mapping[raw] = inst
        overlap = len(wanted.keys() & mapping.keys())
        if not bad and overlap > best_overlap:
            best_field, best_map, best_overlap = field, mapping, overlap
    if best_map is None or best_overlap == 0:
        raise RuntimeError(
            "Synthetic' pickle objects have no original IDs matching Method-2 targets. "
            "Cannot safely join by pickle position. Verify the PKL ID schema."
        )
    missing = wanted.keys() - best_map.keys()
    if missing:
        raise RuntimeError(f"Synthetic' pickle has {len(missing)} unmatched original IDs; "
                           f"ID field={best_field}; examples={sorted(missing)[:20]}")
    print(f"[{_ts()}] Synthetic' graph IDs matched via {best_field}: {len(wanted)}")
    out = {("synthetic", key): best_map[id_num] for id_num, key in wanted.items()}
    if len(out) != EXPECTED_SYNTHETIC_N:
        raise RuntimeError(f"Synthetic' coverage {len(out)} != {EXPECTED_SYNTHETIC_N}")
    return out


# ============================================================
# EXACT FINAL SKLEARN COMBINED SPLIT
# ============================================================

def load_final_combined_split(
    combined_targets: pd.DataFrame,
) -> Tuple[set, set, pd.DataFrame]:
    if not COMBINED_SPLIT_FILE.exists():
        raise FileNotFoundError(
            "Final sklearn split manifest is required so GNN and sklearn "
            "use exactly the same Combined train/test instances.\n"
            f"Missing:\n{COMBINED_SPLIT_FILE}\n\n"
            "Run the current timestamped sklearn Synthetic-prime runner first."
        )

    m = pd.read_csv(
        COMBINED_SPLIT_FILE
    )

    _require_columns(
        m,
        [
            "_key",
            "split",
        ],
        "final sklearn Combined split manifest",
    )

    m["_key"] = (
        m["_key"]
        .astype(str)
        .str.strip()
    )

    m["split"] = (
        m["split"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    _assert_unique(
        m,
        "_key",
        "final sklearn Combined split manifest",
    )

    bad_split = (
        set(m["split"])
        - {"train", "test"}
    )

    if bad_split:
        raise ValueError(
            f"Combined split manifest contains bad split values: {bad_split}"
        )

    target_keys = set(
        combined_targets["_key"]
    )

    manifest_keys = set(
        m["_key"]
    )

    if manifest_keys != target_keys:
        raise RuntimeError(
            "Final sklearn split manifest does not match the final "
            "280-instance Combined target set.\n"
            f"Missing={sorted(target_keys-manifest_keys)[:20]}\n"
            f"Extra={sorted(manifest_keys-target_keys)[:20]}"
        )

    if len(m) != EXPECTED_COMBINED_N:
        raise RuntimeError(
            f"Final sklearn split manifest has {len(m)} rows, "
            f"expected {EXPECTED_COMBINED_N}."
        )

    train_keys = set(
        m.loc[
            m["split"] == "train",
            "_key",
        ]
    )

    test_keys = set(
        m.loc[
            m["split"] == "test",
            "_key",
        ]
    )

    if train_keys & test_keys:
        raise RuntimeError(
            "Sklearn Combined split manifest has train/test overlap."
        )

    print(
        f"[{_ts()}] Reusing FINAL sklearn Combined split: "
        f"train={len(train_keys)} test={len(test_keys)}",
        flush=True,
    )

    return (
        train_keys,
        test_keys,
        m,
    )


# ============================================================
# GRAPH BUILDING — SAME SUPPLIED STATIC TRIPARTITE GNN
# ============================================================

def _safe(arr):
    arr = np.nan_to_num(
        np.asarray(
            arr,
            dtype=float,
        ),
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    return np.clip(
        arr,
        -_CLIP,
        _CLIP,
    )


def _safe_scalar(v):
    if not np.isfinite(
        float(v)
    ):
        return 0.0

    return float(
        np.clip(
            v,
            -_CLIP,
            _CLIP,
        )
    )


def _as_1d(x):
    return np.asarray(
        x,
        dtype=float,
    ).reshape(-1)


def _as_2d(x):
    a = np.asarray(
        x,
        dtype=float,
    )

    if a.ndim != 2:
        raise ValueError(
            f"Expected 2D array, got shape {a.shape}"
        )

    return a


def _symmetrize(A):
    return 0.5 * (
        A + A.T
    )


def _upper_tri_coeffs(A):
    n = A.shape[0]
    iu, ju = np.triu_indices(n)

    c = A[
        iu,
        ju,
    ].astype(float).copy()

    c[iu != ju] *= 2.0

    return c


def _triu_flat_index(
    iu,
    ju,
    n,
):
    iu = iu.astype(
        np.int64
    )

    ju = ju.astype(
        np.int64
    )

    return (
        iu * n
        - (iu * (iu - 1)) // 2
        + (ju - iu)
    ).astype(np.int64)


def _select_top_pairs(
    A_list,
    tol,
    max_quad,
):
    n = A_list[0].shape[0]

    iu_all, ju_all = np.triu_indices(n)

    score = np.zeros(
        len(iu_all),
        dtype=float,
    )

    for Ak in A_list:
        score = np.maximum(
            score,
            np.abs(
                _upper_tri_coeffs(
                    _symmetrize(Ak)
                )
            ),
        )

    keep = (
        score > tol
    )

    iu = (
        iu_all[keep]
        .astype(int)
    )

    ju = (
        ju_all[keep]
        .astype(int)
    )

    if len(iu) == 0:
        return (
            np.array([0], int),
            np.array([0], int),
        )

    if (
        max_quad is not None
        and len(iu) > max_quad
    ):
        sc = score[keep]

        idx = np.argpartition(
            sc,
            -max_quad,
        )[-max_quad:]

        order = np.argsort(
            -sc[idx]
        )

        iu = iu[idx][order]
        ju = ju[idx][order]

    return iu, ju


def _make_edges(
    src,
    dst,
    attr,
):
    if len(src) == 0:
        return (
            torch.empty(
                (2, 0),
                dtype=torch.long,
            ),
            torch.empty(
                (0, 1),
                dtype=torch.float,
            ),
        )

    ei = torch.tensor(
        [src, dst],
        dtype=torch.long,
    )

    ea = torch.tensor(
        attr,
        dtype=torch.float,
    )

    if ea.dim() == 1:
        ea = ea.view(
            -1,
            1,
        )

    return ei, ea


class FeatureNorm:
    """
    Same train-only graph feature normalizer as supplied code.
    Enabled in this stable run because the Combined MINLPLib+QPLIB coefficient scales can otherwise produce float32 activation/loss overflow. Statistics are fit on TRAIN graphs only.
    """
    def __init__(self):
        self.stats = {}
        self.fitted = False

    def _accumulate(
        self,
        store,
        key,
        arr,
    ):
        store.setdefault(
            key,
            [],
        ).append(
            np.asarray(
                arr,
                dtype=float,
            )
        )

    def fit(
        self,
        graphs,
    ):
        store = {}

        for g in graphs:
            self._accumulate(
                store,
                "var",
                g["var"].x.numpy(),
            )

            self._accumulate(
                store,
                "quad",
                g["quad"].x.numpy(),
            )

            self._accumulate(
                store,
                "constr",
                g["constr"].x.numpy(),
            )

            for et in [
                ("var", "inc", "quad"),
                ("quad", "qcoef", "constr"),
                ("var", "lin", "constr"),
            ]:
                ea = g[et].edge_attr

                if ea.numel() > 0:
                    self._accumulate(
                        store,
                        f"edge::{et}",
                        ea.numpy(),
                    )

        for key, chunks in store.items():
            X = np.concatenate(
                chunks,
                axis=0,
            )

            mu = X.mean(axis=0)
            sd = X.std(axis=0)

            sd[
                sd < 1e-8
            ] = 1.0

            self.stats[key] = (
                mu.astype(np.float32),
                sd.astype(np.float32),
            )

        self.fitted = True

    def apply(
        self,
        g,
    ):
        if not self.fitted:
            return g

        def zt(
            key,
            x,
        ):
            if key not in self.stats:
                return x

            mu, sd = self.stats[key]

            mu = torch.tensor(mu)
            sd = torch.tensor(sd)

            return (
                x - mu
            ) / sd

        g["var"].x = zt(
            "var",
            g["var"].x,
        )

        g["quad"].x = zt(
            "quad",
            g["quad"].x,
        )

        g["constr"].x = zt(
            "constr",
            g["constr"].x,
        )

        for et in [
            ("var", "inc", "quad"),
            ("quad", "qcoef", "constr"),
            ("var", "lin", "constr"),
        ]:
            if g[et].edge_attr.numel() > 0:
                g[et].edge_attr = zt(
                    f"edge::{et}",
                    g[et].edge_attr,
                )

        pairs = {
            ("var", "inc", "quad"):
                ("quad", "rev_inc", "var"),
            ("quad", "qcoef", "constr"):
                ("constr", "rev_qcoef", "quad"),
            ("var", "lin", "constr"):
                ("constr", "rev_lin", "var"),
        }

        for fwd, rev in pairs.items():
            g[rev].edge_attr = g[fwd].edge_attr

        return g


def build_graph(
    inst,
    delta: float,
    y_class: int,
) -> HeteroData:
    """
    STATIC graph only.
    No DI features.
    No probe / dynamic features.
    """
    A_list = [
        _as_2d(Ak)
        for Ak in get_field(
            inst,
            "A_list",
        )
    ]

    b_list = [
        _as_1d(bk)
        for bk in get_field(
            inst,
            "b_list",
        )
    ]

    c_list = [
        float(
            np.asarray(ck)
            .reshape(-1)[0]
        )
        for ck in get_field(
            inst,
            "c_list",
        )
    ]

    l = _as_1d(
        get_field(
            inst,
            "l",
        )
    )

    u = _as_1d(
        get_field(
            inst,
            "u",
        )
    )

    n = int(
        l.shape[0]
    )

    m_total = len(
        A_list
    )

    max_q = (
        None
        if MAX_QUAD_NODES <= 0
        else int(MAX_QUAD_NODES)
    )

    max_lin = (
        None
        if MAX_LIN_EDGES <= 0
        else int(MAX_LIN_EDGES)
    )

    data = HeteroData()

    data["var"].x = torch.tensor(
        np.stack(
            [
                _safe(
                    2.0 * b_list[0]
                ),
                _safe(l),
                _safe(u),
                _safe(u) - _safe(l),
                np.ones(n),
            ],
            axis=1,
        ),
        dtype=torch.float,
    )

    cc = _safe(
        np.array(
            [
                _safe_scalar(
                    c_list[k]
                )
                for k in range(
                    1,
                    m_total,
                )
            ]
        )
    )

    data["constr"].x = torch.tensor(
        np.stack(
            [
                cc,
                np.ones_like(cc),
            ],
            axis=1,
        ),
        dtype=torch.float,
    )

    iu_sel, ju_sel = _select_top_pairs(
        A_list,
        TOL,
        max_q,
    )

    sel_idx = _triu_flat_index(
        iu_sel,
        ju_sel,
        n,
    )

    coeff0_sel = _safe(
        _upper_tri_coeffs(
            _symmetrize(
                A_list[0]
            )
        )[sel_idx]
    )

    is_diag = (
        iu_sel == ju_sel
    ).astype(float)

    if PERMUTATION_INVARIANT:
        col3 = np.zeros_like(
            is_diag
        )
        col4 = np.zeros_like(
            is_diag
        )
    else:
        col3 = (
            iu_sel
            / max(
                1.0,
                n - 1,
            )
        )

        col4 = (
            ju_sel
            / max(
                1.0,
                n - 1,
            )
        )

    data["quad"].x = torch.tensor(
        np.stack(
            [
                coeff0_sel,
                is_diag,
                col3,
                col4,
                np.ones_like(is_diag),
            ],
            axis=1,
        ),
        dtype=torch.float,
    )

    # var -> quad
    vq_s, vq_d, vq_a = [], [], []

    for q_id, (i, j) in enumerate(
        zip(
            iu_sel.tolist(),
            ju_sel.tolist(),
        )
    ):
        vq_s.append(int(i))
        vq_d.append(q_id)
        vq_a.append([1.0])

        if int(j) != int(i):
            vq_s.append(int(j))
            vq_d.append(q_id)
            vq_a.append([1.0])

    ei, ea = _make_edges(
        vq_s,
        vq_d,
        vq_a,
    )

    data[
        "var",
        "inc",
        "quad",
    ].edge_index = ei

    data[
        "var",
        "inc",
        "quad",
    ].edge_attr = ea

    rev = (
        torch.stack(
            [
                ei[1],
                ei[0],
            ],
            0,
        )
        if ei.numel() > 0
        else torch.empty(
            (2, 0),
            dtype=torch.long,
        )
    )

    data[
        "quad",
        "rev_inc",
        "var",
    ].edge_index = rev

    data[
        "quad",
        "rev_inc",
        "var",
    ].edge_attr = ea

    # quad -> constr
    qc_s, qc_d, qc_a = [], [], []

    for k in range(
        1,
        m_total,
    ):
        coeffk = _safe(
            _upper_tri_coeffs(
                _symmetrize(
                    A_list[k]
                )
            )[sel_idx]
        )

        for q_id in np.where(
            np.abs(coeffk) > TOL
        )[0]:
            qc_s.append(
                int(q_id)
            )

            qc_d.append(
                k - 1
            )

            qc_a.append(
                [
                    float(
                        coeffk[q_id]
                    )
                ]
            )

    ei, ea = _make_edges(
        qc_s,
        qc_d,
        qc_a,
    )

    data[
        "quad",
        "qcoef",
        "constr",
    ].edge_index = ei

    data[
        "quad",
        "qcoef",
        "constr",
    ].edge_attr = ea

    rev = (
        torch.stack(
            [
                ei[1],
                ei[0],
            ],
            0,
        )
        if ei.numel() > 0
        else torch.empty(
            (2, 0),
            dtype=torch.long,
        )
    )

    data[
        "constr",
        "rev_qcoef",
        "quad",
    ].edge_index = rev

    data[
        "constr",
        "rev_qcoef",
        "quad",
    ].edge_attr = ea

    # var -> constr
    vc_s, vc_d, vc_a = [], [], []

    for k in range(
        1,
        m_total,
    ):
        bk = _safe(
            2.0
            * b_list[k]
        )

        keep = (
            np.arange(n)
            if (
                max_lin is None
                or max_lin >= n
            )
            else np.argpartition(
                np.abs(bk),
                -max_lin,
            )[-max_lin:]
        )

        for i in keep:
            val = float(
                bk[int(i)]
            )

            if abs(val) > TOL:
                vc_s.append(
                    int(i)
                )

                vc_d.append(
                    k - 1
                )

                vc_a.append(
                    [val]
                )

    ei, ea = _make_edges(
        vc_s,
        vc_d,
        vc_a,
    )

    data[
        "var",
        "lin",
        "constr",
    ].edge_index = ei

    data[
        "var",
        "lin",
        "constr",
    ].edge_attr = ea

    rev = (
        torch.stack(
            [
                ei[1],
                ei[0],
            ],
            0,
        )
        if ei.numel() > 0
        else torch.empty(
            (2, 0),
            dtype=torch.long,
        )
    )

    data[
        "constr",
        "rev_lin",
        "var",
    ].edge_index = rev

    data[
        "constr",
        "rev_lin",
        "var",
    ].edge_attr = ea

    if USE_SIZE_IN_GLOBAL:
        gx = [
            _safe_scalar(
                c_list[0]
            ),
            float(n) / 500.0,
            float(m_total - 1) / 500.0,
        ]
    else:
        # Same width=3 behavior as supplied code.
        gx = [
            _safe_scalar(
                c_list[0]
            ),
            0.0,
            0.0,
        ]

    data.global_x = torch.tensor(
        [gx],
        dtype=torch.float,
    )

    # Use the already-computed final Method-2 target directly.
    data.delta = torch.tensor(
        [float(delta)],
        dtype=torch.float,
    )

    data.choice_mask = torch.tensor(
        [1.0],
        dtype=torch.float,
    )

    data.finite_mask = torch.tensor(
        [1.0],
        dtype=torch.float,
    )

    data.y_class = torch.tensor(
        [int(y_class)],
        dtype=torch.long,
    )

    data.y_direct = torch.tensor(
        [float(y_class)],
        dtype=torch.float,
    )

    data.label_mask = torch.tensor(
        [1.0],
        dtype=torch.float,
    )

    data.delta_raw_for_weight = torch.tensor(
        [float(delta)],
        dtype=torch.float,
    )

    return data


# ============================================================
# DATASET
# ============================================================

class QCQPDataset(
    torch.utils.data.Dataset
):
    def __init__(
        self,
        instances_by_key: Dict[
            Tuple[str, str],
            Any,
        ],
        df: pd.DataFrame,
        norm: Optional[
            FeatureNorm
        ],
        name: str,
    ):
        self.instances_by_key = (
            instances_by_key
        )

        self.df = (
            df.reset_index(
                drop=True
            )
        )

        self.norm = norm
        self.name = str(name)

        print(
            f"[{_ts()}] [{self.name}] "
            f"dataset size={len(self.df)}",
            flush=True,
        )

    def __len__(self):
        return len(
            self.df
        )

    def build_raw(
        self,
        idx,
    ):
        row = self.df.iloc[
            idx
        ]

        dataset_kind = str(
            row[
                "_dataset_kind"
            ]
        )

        key = str(
            row["_key"]
        )

        inst_key = (
            dataset_kind,
            key,
        )

        if inst_key not in self.instances_by_key:
            raise KeyError(
                "Missing QCQP instance for "
                f"{inst_key}"
            )

        inst = (
            self.instances_by_key[
                inst_key
            ]
        )

        delta = float(
            row[TARGET_COL]
        )

        y_class = int(
            row[CLASS_COL]
        )

        return build_graph(
            inst,
            delta=delta,
            y_class=y_class,
        )

    def __getitem__(
        self,
        idx,
    ):
        g = self.build_raw(
            idx
        )

        if self.norm is not None:
            g = self.norm.apply(
                g
            )

        return g


# ============================================================
# MODEL — SAME STATIC TRIPARTITE GNN
# ============================================================

class MLP(nn.Module):
    def __init__(
        self,
        i,
        o,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(i, o),
            nn.ReLU(),
            nn.Linear(o, o),
        )

    def forward(
        self,
        x,
    ):
        return self.net(
            x
        )


def _ews(
    x_src,
    ei,
    ea,
    num_dst,
):
    if ei.numel() == 0:
        return x_src.new_zeros(
            (
                num_dst,
                x_src.size(1),
            )
        )

    return scatter(
        x_src[
            ei[0]
        ]
        * ea,
        ei[1],
        dim=0,
        dim_size=num_dst,
        reduce="sum",
    )


class TripartiteGNN(nn.Module):
    def __init__(
        self,
        hidden=64,
        layers=3,
        dropout=0.01,
        two_heads=True,
    ):
        super().__init__()

        H = hidden

        self.hidden = hidden
        self.layers = layers
        self.dropout = dropout
        self.two_heads = two_heads

        self.lin_var = nn.Linear(
            5,
            H,
        )

        self.lin_quad = nn.Linear(
            5,
            H,
        )

        self.lin_con = nn.Linear(
            2,
            H,
        )

        mk = lambda: nn.ModuleList(
            [
                nn.Linear(
                    H,
                    H,
                )
                for _ in range(
                    layers
                )
            ]
        )

        self.f_v_to_q = mk()
        self.f_v_to_c = mk()
        self.f_q_to_c = mk()
        self.f_c_to_q = mk()
        self.f_c_to_v = mk()
        self.f_q_to_v = mk()

        self.g_q1 = nn.ModuleList(
            [
                MLP(
                    H * 2,
                    H,
                )
                for _ in range(
                    layers
                )
            ]
        )

        self.g_c = nn.ModuleList(
            [
                MLP(
                    H * 3,
                    H,
                )
                for _ in range(
                    layers
                )
            ]
        )

        self.g_q2 = nn.ModuleList(
            [
                MLP(
                    H * 2,
                    H,
                )
                for _ in range(
                    layers
                )
            ]
        )

        self.g_v = nn.ModuleList(
            [
                MLP(
                    H * 3,
                    H,
                )
                for _ in range(
                    layers
                )
            ]
        )

        # EXACTLY the original static graph head width:
        # pooled var + pooled quad + pooled constr + global_x(3)
        head_in = (
            H * 3
            + 3
        )

        def _head():
            return nn.Sequential(
                nn.Linear(
                    head_in,
                    H,
                ),
                nn.ReLU(),
                nn.Dropout(
                    dropout
                ),
                nn.Linear(
                    H,
                    1,
                ),
            )

        self.head_direct = _head()

        self.head_delta = (
            _head()
            if two_heads
            else None
        )

    def forward(
        self,
        data,
    ):
        h_v = self.lin_var(
            data[
                "var"
            ].x
        )

        h_q = self.lin_quad(
            data[
                "quad"
            ].x
        )

        h_c = self.lin_con(
            data[
                "constr"
            ].x
        )

        e_vq = data[
            "var",
            "inc",
            "quad",
        ].edge_index

        w_vq = data[
            "var",
            "inc",
            "quad",
        ].edge_attr

        e_vc = data[
            "var",
            "lin",
            "constr",
        ].edge_index

        w_vc = data[
            "var",
            "lin",
            "constr",
        ].edge_attr

        e_qc = data[
            "quad",
            "qcoef",
            "constr",
        ].edge_index

        w_qc = data[
            "quad",
            "qcoef",
            "constr",
        ].edge_attr

        e_cq = data[
            "constr",
            "rev_qcoef",
            "quad",
        ].edge_index

        w_cq = data[
            "constr",
            "rev_qcoef",
            "quad",
        ].edge_attr

        e_cv = data[
            "constr",
            "rev_lin",
            "var",
        ].edge_index

        w_cv = data[
            "constr",
            "rev_lin",
            "var",
        ].edge_attr

        e_qv = data[
            "quad",
            "rev_inc",
            "var",
        ].edge_index

        w_qv = data[
            "quad",
            "rev_inc",
            "var",
        ].edge_attr

        nv = data[
            "var"
        ].num_nodes

        nq = data[
            "quad"
        ].num_nodes

        nc = data[
            "constr"
        ].num_nodes

        for t in range(
            self.layers
        ):
            h_q_bar = self.g_q1[t](
                torch.cat(
                    [
                        h_q,
                        _ews(
                            self.f_v_to_q[t](
                                h_v
                            ),
                            e_vq,
                            w_vq,
                            nq,
                        ),
                    ],
                    1,
                )
            )

            h_c_new = self.g_c[t](
                torch.cat(
                    [
                        h_c,
                        _ews(
                            self.f_v_to_c[t](
                                h_v
                            ),
                            e_vc,
                            w_vc,
                            nc,
                        ),
                        _ews(
                            self.f_q_to_c[t](
                                h_q_bar
                            ),
                            e_qc,
                            w_qc,
                            nc,
                        ),
                    ],
                    1,
                )
            )

            h_q_new = self.g_q2[t](
                torch.cat(
                    [
                        h_q_bar,
                        _ews(
                            self.f_c_to_q[t](
                                h_c_new
                            ),
                            e_cq,
                            w_cq,
                            nq,
                        ),
                    ],
                    1,
                )
            )

            h_v_new = self.g_v[t](
                torch.cat(
                    [
                        h_v,
                        _ews(
                            self.f_c_to_v[t](
                                h_c_new
                            ),
                            e_cv,
                            w_cv,
                            nv,
                        ),
                        _ews(
                            self.f_q_to_v[t](
                                h_q_new
                            ),
                            e_qv,
                            w_qv,
                            nv,
                        ),
                    ],
                    1,
                )
            )

            h_v = F.dropout(
                F.relu(
                    h_v_new
                ),
                p=self.dropout,
                training=self.training,
            )

            h_q = F.dropout(
                F.relu(
                    h_q_new
                ),
                p=self.dropout,
                training=self.training,
            )

            h_c = F.dropout(
                F.relu(
                    h_c_new
                ),
                p=self.dropout,
                training=self.training,
            )

        B = (
            int(
                data[
                    "var"
                ]
                .batch
                .max()
                .item()
            )
            + 1
        )

        pv = global_mean_pool(
            h_v,
            data[
                "var"
            ].batch,
            size=B,
        )

        pq = (
            global_mean_pool(
                h_q,
                data[
                    "quad"
                ].batch,
                size=B,
            )
            if nq > 0
            else h_v.new_zeros(
                (
                    B,
                    self.hidden,
                )
            )
        )

        pc = (
            global_mean_pool(
                h_c,
                data[
                    "constr"
                ].batch,
                size=B,
            )
            if nc > 0
            else h_v.new_zeros(
                (
                    B,
                    self.hidden,
                )
            )
        )

        gx = (
            data.global_x
        )

        if gx.dim() == 1:
            gx = gx.view(
                -1,
                3,
            )

        if gx.size(0) != B:
            gx = gx.view(
                B,
                -1,
            )

        g = torch.cat(
            [
                pv,
                pq,
                pc,
                gx,
            ],
            dim=1,
        )

        ld = self.head_direct(
            g
        ).view(-1)

        lk = (
            self.head_delta(
                g
            ).view(-1)
            if self.two_heads
            else ld
        )

        return (
            ld,
            lk,
        )


# ============================================================
# TRAIN / EVAL — PRESERVED FROM SUPPLIED GNN
# ============================================================

def pos_weight_from_df(
    df: pd.DataFrame,
    device,
):
    y = (
        pd.to_numeric(
            df[CLASS_COL],
            errors="raise",
        )
        .astype(int)
        .to_numpy()
    )

    n0 = int(
        (y == 0).sum()
    )

    n1 = int(
        (y == 1).sum()
    )

    if n1 == 0:
        return None

    return torch.tensor(
        [
            float(n0)
            / float(
                n1 + 1e-12
            )
        ],
        dtype=torch.float,
        device=device,
    )


def train_one_epoch(
    model,
    loader,
    opt,
    device,
    pw,
    task,
):
    model.train()

    tot = 0.0
    n = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        batch = batch.to(
            device
        )

        opt.zero_grad()

        ld, lk = model(
            batch
        )

        d = (
            batch.delta
            .view(-1)
            .float()
        )

        m = (
            batch.choice_mask
            .view(-1)
            .float()
        )

        w = (
            1.0
            - WEIGHT_ALPHA
        ) + (
            WEIGHT_ALPHA
            * d.abs()
        )

        if m.sum().item() == 0:
            continue

        if task == "classification":
            yd = (
                batch.y_direct
                .view(-1)
                .float()
            )

            lm = (
                batch.label_mask
                .view(-1)
                .float()
            )

            if lm.sum().item() == 0:
                continue

            bce = F.binary_cross_entropy_with_logits(
                ld,
                yd,
                pos_weight=pw,
                reduction="none",
            )

            loss_dir = (
                (
                    bce
                    * w
                    * lm
                ).sum()
                / (
                    (
                        w
                        * lm
                    ).sum()
                    + 1e-12
                )
            )

            lk_s = torch.where(
                torch.isfinite(
                    lk
                ),
                lk,
                torch.zeros_like(
                    lk
                ),
            )

            wd = (
                w
                * m
            )

            loss_del = (
                (
                    (
                        lk_s
                        - d
                    ) ** 2
                    * wd
                ).sum()
                / (
                    wd.sum()
                    + 1e-12
                )
            )

            loss = (
                loss_dir
                + LAMBDA_DELTA
                * loss_del
            )

            active_n = float(
                lm.sum().item()
            )

        else:
            pred = torch.where(
                torch.isfinite(
                    ld
                ),
                ld,
                torch.zeros_like(
                    ld
                ),
            )

            if USE_SAMPLE_WEIGHTS:
                wd = (
                    m
                    * w
                )

                loss = (
                    (
                        (
                            pred
                            - d
                        ) ** 2
                        * wd
                    ).sum()
                    / (
                        wd.sum()
                        + 1e-12
                    )
                )
            else:
                loss = (
                    (
                        (
                            pred
                            - d
                        ) ** 2
                        * m
                    ).sum()
                    / (
                        m.sum()
                        + 1e-12
                    )
                )

            if USE_DECISION_AUX:
                y_dec = (
                    d
                    > -TAU_SIGN
                ).float()

                logit = (
                    DECISION_SCALE
                    * (
                        pred
                        + TAU_SIGN
                    )
                )

                bce = F.binary_cross_entropy_with_logits(
                    logit,
                    y_dec,
                    pos_weight=pw,
                    reduction="none",
                )

                loss = (
                    loss
                    + DECISION_LOSS_WEIGHT
                    * (
                        (
                            bce
                            * m
                        ).sum()
                        / (
                            m.sum()
                            + 1e-12
                        )
                    )
                )

            active_n = float(
                m.sum().item()
            )

        if not torch.isfinite(
            loss
        ):
            def _mx(t):
                if t is None or t.numel() == 0:
                    return float("nan")
                tf = t.detach()
                finite = tf[torch.isfinite(tf)]
                return float(finite.abs().max().item()) if finite.numel() else float("nan")

            raise RuntimeError(
                "Non-finite GNN training loss encountered. "
                f"task={task}, batch={batch_idx}, "
                f"max|ld|={_mx(ld):.6e}, max|lk|={_mx(lk):.6e}, "
                f"max|delta|={_mx(d):.6e}, "
                f"ld_all_finite={bool(torch.isfinite(ld).all().item())}, "
                f"lk_all_finite={bool(torch.isfinite(lk).all().item())}. "
                "FeatureNorm is ON in this stable script; if this still occurs, "
                "send this exact diagnostic line."
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0,
        )

        opt.step()

        tot += (
            float(
                loss.item()
            )
            * active_n
        )

        n += active_n

    return (
        tot
        / max(
            1.0,
            n,
        )
    )


@torch.no_grad()
def predict(
    model,
    loader,
    device,
    task,
):
    model.eval()

    D = []
    P = []
    Y = []
    M = []
    PROB = []
    DELTA_PRED = []

    for batch in loader:
        batch = batch.to(
            device
        )

        ld, lk = model(
            batch
        )

        D.append(
            batch.delta
            .view(-1)
            .cpu()
            .numpy()
        )

        Y.append(
            batch.y_class
            .view(-1)
            .cpu()
            .numpy()
        )

        M.append(
            batch.choice_mask
            .view(-1)
            .cpu()
            .numpy()
        )

        if task == "classification":
            ld_s = torch.where(
                torch.isfinite(
                    ld
                ),
                ld,
                torch.zeros_like(
                    ld
                ),
            )

            prob = torch.sigmoid(
                ld_s
            )

            y_hat = (
                prob >= 0.5
            ).float()

            lk_s = torch.where(
                torch.isfinite(
                    lk
                ),
                lk,
                torch.zeros_like(
                    lk
                ),
            )

            P.append(
                y_hat
                .cpu()
                .numpy()
            )

            PROB.append(
                prob
                .cpu()
                .numpy()
            )

            DELTA_PRED.append(
                lk_s
                .cpu()
                .numpy()
            )

        else:
            pred = torch.where(
                torch.isfinite(
                    ld
                ),
                ld,
                torch.zeros_like(
                    ld
                ),
            )

            P.append(
                pred
                .cpu()
                .numpy()
            )

            PROB.append(
                torch.full_like(
                    pred,
                    float("nan"),
                )
                .cpu()
                .numpy()
            )

            DELTA_PRED.append(
                pred
                .cpu()
                .numpy()
            )

    return (
        np.concatenate(D),
        np.concatenate(P),
        np.concatenate(Y),
        np.concatenate(M),
        np.concatenate(PROB),
        np.concatenate(DELTA_PRED),
    )


def safe_pearson_r(
    y_true,
    y_pred,
):
    yt = np.asarray(
        y_true,
        dtype=float,
    )

    yp = np.asarray(
        y_pred,
        dtype=float,
    )

    mask = (
        np.isfinite(
            yt
        )
        & np.isfinite(
            yp
        )
    )

    yt = yt[mask]
    yp = yp[mask]

    if (
        len(yt) < 2
        or np.std(yt) <= 0
        or np.std(yp) <= 0
    ):
        return float(
            "nan"
        )

    r = np.corrcoef(
        yt,
        yp,
    )[0, 1]

    return (
        float(r)
        if np.isfinite(r)
        else float("nan")
    )


def r_accuracy_paper(
    yt,
    yp,
    eps=TAU_SIGN,
):
    yt = np.asarray(
        yt,
        float,
    )

    yp = np.asarray(
        yp,
        float,
    )

    mask = (
        np.isfinite(
            yt
        )
        & np.isfinite(
            yp
        )
    )

    yt = yt[mask]
    yp = yp[mask]

    if len(yt) == 0:
        return float(
            "nan"
        )

    same = (
        np.sign(yt)
        == np.sign(yp)
    )

    forgiveness = (
        (yt > -eps)
        & (yt < 0)
        & (yp > 0)
    )

    return float(
        np.mean(
            same
            | forgiveness
        )
    )


def r_accuracy_combined(
    yt,
    yp,
    eps=TAU_SIGN,
):
    yt = np.asarray(
        yt,
        float,
    )

    yp = np.asarray(
        yp,
        float,
    )

    mask = (
        np.isfinite(
            yt
        )
        & np.isfinite(
            yp
        )
    )

    yt = yt[mask]
    yp = yp[mask]

    if len(yt) == 0:
        return float(
            "nan"
        )

    same = (
        np.sign(yt)
        == np.sign(yp)
    )

    q2 = (
        (yt > -eps)
        & (yt < 0)
        & (yp > 0)
    )

    q4 = (
        (yt > 0)
        & (yt < eps)
        & (yp < 0)
        & (yp > -eps)
    )

    return float(
        np.mean(
            same
            | q2
            | q4
        )
    )


def decision_accuracy_against_y(
    y_true_class,
    y_pred_delta,
    tau=TAU_SIGN,
):
    y_pred = (
        np.asarray(
            y_pred_delta,
            dtype=float,
        )
        > -tau
    ).astype(int)

    return float(
        np.mean(
            np.asarray(
                y_true_class,
                dtype=int,
            )
            == y_pred
        )
    )


# ============================================================
# OUTPUT HELPERS
# ============================================================

def save_classification_outputs(
    run_dir: Path,
    test_df: pd.DataFrame,
    y_true,
    y_pred,
    y_prob,
    delta_pred,
) -> None:
    out = test_df[
        [
            "_dataset_kind",
            "_key",
        ]
    ].copy()

    out.insert(
        0,
        "split",
        "test",
    )

    out["y_true"] = np.asarray(
        y_true,
        dtype=int,
    )

    out["y_pred"] = np.asarray(
        y_pred,
        dtype=int,
    )

    out["y_prob_LP"] = np.asarray(
        y_prob,
        dtype=float,
    )

    out["delta_true"] = (
        test_df[
            TARGET_COL
        ]
        .to_numpy(
            float
        )
    )

    out["delta_pred_aux"] = np.asarray(
        delta_pred,
        dtype=float,
    )

    out["selected_relaxation"] = np.where(
        out["y_pred"] == 1,
        "LP",
        "SDP_PRIME",
    )

    out.to_csv(
        run_dir
        / "preds_test.csv",
        index=False,
    )


def save_regression_outputs(
    run_dir: Path,
    test_df: pd.DataFrame,
    delta_true,
    delta_pred,
    y_true,
) -> None:
    y_pred = (
        np.asarray(
            delta_pred,
            dtype=float,
        )
        > -TAU_SIGN
    ).astype(int)

    out = test_df[
        [
            "_dataset_kind",
            "_key",
        ]
    ].copy()

    out.insert(
        0,
        "split",
        "test",
    )

    out["delta_true"] = np.asarray(
        delta_true,
        dtype=float,
    )

    out["delta_pred"] = np.asarray(
        delta_pred,
        dtype=float,
    )

    out["y_true"] = np.asarray(
        y_true,
        dtype=int,
    )

    out["y_pred_from_delta"] = (
        y_pred
    )

    out["selected_relaxation"] = np.where(
        y_pred == 1,
        "LP",
        "SDP_PRIME",
    )

    out.to_csv(
        run_dir
        / "preds_test.csv",
        index=False,
    )


# ============================================================
# SINGLE EXPERIMENT
# ============================================================

def run_one_experiment(
    *,
    run_id: int,
    train_setting: str,
    task: str,
    combined_targets: pd.DataFrame,
    synthetic_targets: pd.DataFrame,
    combined_train_keys: set,
    combined_test_keys: set,
    instances_by_key: Dict[
        Tuple[str, str],
        Any,
    ],
    device,
) -> Dict[str, Any]:
    task = task.lower()

    if train_setting == "combined_to_combined":
        train_df = (
            combined_targets[
                combined_targets[
                    "_key"
                ].isin(
                    combined_train_keys
                )
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )

        test_df = (
            combined_targets[
                combined_targets[
                    "_key"
                ].isin(
                    combined_test_keys
                )
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )

        train_df[
            "_dataset_kind"
        ] = "combined"

        test_df[
            "_dataset_kind"
        ] = "combined"

        train_test_label = (
            "Combined→Combined"
        )

    elif train_setting == "synthetic_to_combined":
        train_df = (
            synthetic_targets
            .copy()
            .reset_index(
                drop=True
            )
        )

        test_df = (
            combined_targets
            .copy()
            .reset_index(
                drop=True
            )
        )

        train_df[
            "_dataset_kind"
        ] = "synthetic"

        test_df[
            "_dataset_kind"
        ] = "combined"

        train_test_label = (
            "Synthetic′→Combined"
        )

    else:
        raise ValueError(
            train_setting
        )

    for name, df in [
        ("train", train_df),
        ("test", test_df),
    ]:
        if not np.isfinite(
            df[
                TARGET_COL
            ].to_numpy(
                float
            )
        ).all():
            raise RuntimeError(
                f"{name}: non-finite delta."
            )

        yy = (
            df[
                CLASS_COL
            ].to_numpy(
                int
            )
        )

        if not np.isin(
            yy,
            [0, 1],
        ).all():
            raise RuntimeError(
                f"{name}: invalid y."
            )

    # Deterministic initialization per experiment.
    np.random.seed(0)
    torch.manual_seed(0)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            0
        )

    variant = (
        "perm_inv"
        if PERMUTATION_INVARIANT
        else "index_feat"
    )

    run_name = (
        f"run_{run_id:02d}__"
        f"{train_setting}__"
        f"{task}"
    )

    run_dir = (
        OUTPUT_ROOT
        / run_name
    )

    run_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    print(
        "\n"
        + "=" * 90
    )

    print(
        f"[{_ts()}] RUN {run_id:02d} | "
        f"{train_test_label} | STATIC GNN | "
        f"{task.upper()}"
    )

    print(
        "=" * 90
    )

    print(
        f"[{_ts()}] train={len(train_df)} "
        f"test={len(test_df)} "
        f"epochs={EPOCHS}",
        flush=True,
    )

    print(
        f"[{_ts()}] train labels="
        f"{train_df[CLASS_COL].value_counts().sort_index().to_dict()} "
        f"| test labels="
        f"{test_df[CLASS_COL].value_counts().sort_index().to_dict()}",
        flush=True,
    )

    # Save exact train/test membership for the run.
    split_manifest = pd.concat(
        [
            train_df[
                [
                    "_dataset_kind",
                    "_key",
                    CLASS_COL,
                    TARGET_COL,
                ]
            ].assign(
                split="train"
            ),
            test_df[
                [
                    "_dataset_kind",
                    "_key",
                    CLASS_COL,
                    TARGET_COL,
                ]
            ].assign(
                split="test"
            ),
        ],
        ignore_index=True,
    )

    split_manifest.to_csv(
        run_dir
        / "instance_split.csv",
        index=False,
    )

    norm = (
        FeatureNorm()
        if USE_FEATURE_NORM
        else None
    )

    if norm is not None:
        tmp = QCQPDataset(
            instances_by_key,
            train_df,
            norm=None,
            name=(
                f"run{run_id:02d}_"
                "train_raw"
            ),
        )

        norm.fit(
            [
                tmp.build_raw(i)
                for i in range(
                    len(tmp)
                )
            ]
        )

        print(
            f"[{_ts()}] feature norm fitted on "
            f"{len(tmp)} train graphs",
            flush=True,
        )

    train_ds = QCQPDataset(
        instances_by_key,
        train_df,
        norm,
        name=(
            f"run{run_id:02d}_"
            "train"
        ),
    )

    test_ds = QCQPDataset(
        instances_by_key,
        test_df,
        norm,
        name=(
            f"run{run_id:02d}_"
            "test"
        ),
    )

    tr_loader = DataLoader(
        train_ds,
        batch_size=BATCH_TRAIN,
        shuffle=True,
    )

    te_loader = DataLoader(
        test_ds,
        batch_size=BATCH_EVAL,
        shuffle=False,
    )

    two_heads = (
        task
        == "classification"
    )

    model = TripartiteGNN(
        HIDDEN,
        LAYERS,
        DROPOUT,
        two_heads=two_heads,
    ).to(
        device
    )

    opt = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WD,
    )

    pw = pos_weight_from_df(
        train_df,
        device,
    )

    print(
        f"[{_ts()}] device={device} "
        f"pos_weight="
        f"{None if pw is None else round(float(pw.item()), 6)}",
        flush=True,
    )

    epoch_log = []

    for ep in range(
        1,
        EPOCHS + 1,
    ):
        loss = train_one_epoch(
            model,
            tr_loader,
            opt,
            device,
            pw,
            task,
        )

        epoch_log.append(
            {
                "epoch": ep,
                "train_loss": loss,
            }
        )

        print(
            f"[{_ts()}] Epoch {ep:03d}/{EPOCHS} "
            f"| loss={loss:.6f}",
            flush=True,
        )

    pd.DataFrame(
        epoch_log
    ).to_csv(
        run_dir
        / "epoch_log.csv",
        index=False,
    )

    (
        d_te,
        p_te,
        y_te,
        m_te,
        prob_te,
        delta_head_te,
    ) = predict(
        model,
        te_loader,
        device,
        task,
    )

    lab = (
        m_te == 1
    )

    row: Dict[
        str,
        Any,
    ] = {
        "run_id": run_id,
        "train_test": train_test_label,
        "input": "static_tripartite_graph_only",
        "task": task,
        "n_train": len(train_ds),
        "n_test": len(test_ds),
        "variant": variant,
        "feature_norm": USE_FEATURE_NORM,
        "size_in_global": USE_SIZE_IN_GLOBAL,
        "epochs": EPOCHS,
        "early_stopping": False,
        "validation_split": False,
        "target_probe_filename": TARGET_PROBE,
        "uses_probe_features": False,
        "uses_DI_features": False,
    }

    if task == "classification":
        yc = (
            y_te[lab]
            .astype(int)
        )

        pc = (
            p_te[lab]
            .astype(int)
        )

        acc = (
            accuracy_score(
                yc,
                pc,
            )
            if len(yc)
            else float("nan")
        )

        bal = (
            balanced_accuracy_score(
                yc,
                pc,
            )
            if len(
                np.unique(
                    yc
                )
            ) > 1
            else float("nan")
        )

        f1m = (
            f1_score(
                yc,
                pc,
                average="macro",
                zero_division=0,
            )
            if len(yc)
            else float("nan")
        )

        cm = confusion_matrix(
            yc,
            pc,
            labels=[
                0,
                1,
            ],
        )

        pd.DataFrame(
            cm,
            index=[
                "true_SDP_PRIME(0)",
                "true_LP(1)",
            ],
            columns=[
                "pred_SDP_PRIME(0)",
                "pred_LP(1)",
            ],
        ).to_csv(
            run_dir
            / "cm.csv",
            index=True,
        )

        fig, ax = plt.subplots(
            figsize=(
                8,
                6,
            )
        )

        ConfusionMatrixDisplay(
            cm,
            display_labels=[
                "SDP_PRIME",
                "LP",
            ],
        ).plot(
            ax=ax
        )

        ax.set_title(
            f"{train_test_label} | Static GNN-C"
        )

        plt.tight_layout()

        plt.savefig(
            run_dir
            / "cm.png",
            dpi=200,
        )

        plt.close()

        save_classification_outputs(
            run_dir,
            test_df,
            y_true=y_te,
            y_pred=p_te,
            y_prob=prob_te,
            delta_pred=delta_head_te,
        )

        row.update(
            {
                "accuracy": float(acc),
                "balanced_accuracy": float(bal),
                "f1_macro": float(f1m),
                "r_accuracy_paper": np.nan,
                "r_accuracy_combined": np.nan,
                "rmse": np.nan,
                "mae": np.nan,
                "pearson_r": np.nan,
                "decision_accuracy": np.nan,
            }
        )

        print(
            f"[{_ts()}] GNN-C: "
            f"accuracy={acc:.4f} "
            f"bal_acc={bal:.4f} "
            f"f1_macro={f1m:.4f}",
            flush=True,
        )

    else:
        dt = d_te
        dp = p_te
        yc = (
            y_te
            .astype(int)
        )

        rmse = float(
            np.sqrt(
                mean_squared_error(
                    dt,
                    dp,
                )
            )
        )

        mae = float(
            mean_absolute_error(
                dt,
                dp,
            )
        )

        pearson = safe_pearson_r(
            dt,
            dp,
        )

        r_paper = r_accuracy_paper(
            dt,
            dp,
        )

        r_comb = r_accuracy_combined(
            dt,
            dp,
        )

        dec = decision_accuracy_against_y(
            yc,
            dp,
        )

        save_regression_outputs(
            run_dir,
            test_df,
            delta_true=dt,
            delta_pred=dp,
            y_true=yc,
        )

        fig, ax = plt.subplots(
            figsize=(
                5.5,
                5.5,
            )
        )

        ax.scatter(
            dt,
            dp,
            s=10,
            alpha=0.55,
        )

        lo = min(
            float(
                np.min(dt)
            ),
            float(
                np.min(dp)
            ),
        )

        hi = max(
            float(
                np.max(dt)
            ),
            float(
                np.max(dp)
            ),
        )

        ax.plot(
            [
                lo,
                hi,
            ],
            [
                lo,
                hi,
            ],
            "r--",
            label="Perfect",
        )

        ax.axhline(
            -TAU_SIGN,
            linestyle=":",
            alpha=0.6,
        )

        ax.axvline(
            -TAU_SIGN,
            linestyle=":",
            alpha=0.6,
        )

        ax.set_xlabel(
            "True delta"
        )

        ax.set_ylabel(
            "Predicted delta"
        )

        ax.set_title(
            f"{train_test_label} | Static GNN-R"
        )

        plt.tight_layout()

        plt.savefig(
            run_dir
            / "scatter_test.png",
            dpi=200,
        )

        plt.close()

        y_pred_dec = (
            np.asarray(
                dp
            )
            > -TAU_SIGN
        ).astype(int)

        cm = confusion_matrix(
            yc,
            y_pred_dec,
            labels=[
                0,
                1,
            ],
        )

        pd.DataFrame(
            cm,
            index=[
                "true_SDP_PRIME(0)",
                "true_LP(1)",
            ],
            columns=[
                "pred_SDP_PRIME(0)",
                "pred_LP(1)",
            ],
        ).to_csv(
            run_dir
            / "decision_cm.csv",
            index=True,
        )

        row.update(
            {
                "accuracy": np.nan,
                "balanced_accuracy": np.nan,
                "f1_macro": np.nan,
                "r_accuracy_paper": float(
                    r_paper
                ),
                "r_accuracy_combined": float(
                    r_comb
                ),
                "rmse": float(
                    rmse
                ),
                "mae": float(
                    mae
                ),
                "pearson_r": float(
                    pearson
                ),
                "decision_accuracy": float(
                    dec
                ),
            }
        )

        print(
            f"[{_ts()}] GNN-R: "
            f"rAccuracy_paper={r_paper:.4f} "
            f"rAccuracy_combined={r_comb:.4f} "
            f"RMSE={rmse:.5f} "
            f"MAE={mae:.5f}",
            flush=True,
        )

    pd.DataFrame(
        [row]
    ).to_csv(
        run_dir
        / "results_gnn.csv",
        index=False,
    )

    torch.save(
        model.state_dict(),
        run_dir
        / "model.pt",
    )

    return row


# ============================================================
# PREFLIGHT / OUTPUT RESET
# ============================================================

def preflight_required_paths() -> None:
    required = [
        COMBINED_SPLIT_FILE,
        SYNTHETIC_PKL,
        combined_target_path(
            TARGET_PROBE
        ),
        synthetic_target_path(
            TARGET_PROBE
        ),
    ]

    missing = [
        p
        for p in required
        if not Path(p).exists()
    ]

    for _, d in COMBINED_PKL_DIRS:
        if not Path(d).exists():
            missing.append(
                Path(d)
            )

    if missing:
        raise FileNotFoundError(
            "GNN preflight failed. Missing required paths:\n"
            + "\n".join(
                str(x)
                for x in missing
            )
        )

    print(
        f"[{_ts()}] Preflight OK.",
        flush=True,
    )


def reset_output_root() -> None:
    """Create a fresh timestamped run. Never delete historical results."""
    if OUTPUT_ROOT.exists():
        raise FileExistsError(f"Refusing to overwrite existing results: {OUTPUT_ROOT}")
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
    print(f"[{_ts()}] Fresh GNN results folder: {OUTPUT_ROOT}")


# ============================================================
# MAIN — EXACTLY FOUR GNN EXPERIMENTS
# ============================================================

def main() -> None:
    print(
        "=" * 90
    )

    print(
        f"[{_ts()}] FINAL STATIC-GNN RUNNER"
    )

    print(
        f"[{_ts()}] EXACTLY 4 experiments."
    )

    print(
        f"[{_ts()}] NO dynamic GNN."
    )

    print(
        f"[{_ts()}] NO DI features as GNN input."
    )

    print(
        f"[{_ts()}] NO node-count filter."
    )

    print(
        f"[{_ts()}] Combined expected n="
        f"{EXPECTED_COMBINED_N}"
    )

    print(
        f"[{_ts()}] Synthetic' expected n="
        f"{EXPECTED_SYNTHETIC_N}"
    )

    print(
        f"[{_ts()}] Combined exclusions: "
        f"{sorted(EXCLUDE_COMBINED_KEYS)}"
    )

    print(
        f"[{_ts()}] Target file uses probe{TARGET_PROBE} name only; "
        "probe features are NOT loaded."
    )

    print(
        "=" * 90
    )

    preflight_required_paths()

    # --------------------------------------------------------
    # FINAL TARGETS
    # --------------------------------------------------------
    (
        combined_targets,
        removed_combined,
    ) = load_targets(
        "combined",
        TARGET_PROBE,
    )

    (
        synthetic_targets,
        removed_synthetic,
    ) = load_targets(
        "synthetic",
        TARGET_PROBE,
    )

    # Optional proof of user's statement: target values are the same
    # under probe10/probe20 naming.
    audit_probe10_probe20_targets(
        "combined",
        combined_targets,
    )

    audit_probe10_probe20_targets(
        "synthetic",
        synthetic_targets,
    )

    # --------------------------------------------------------
    # EXACT SKLEARN SPLIT
    # --------------------------------------------------------
    (
        combined_train_keys,
        combined_test_keys,
        split_manifest,
    ) = load_final_combined_split(
        combined_targets
    )

    # --------------------------------------------------------
    # GRAPH INSTANCE COVERAGE
    # --------------------------------------------------------
    combined_target_keys = set(
        combined_targets[
            "_key"
        ]
    )

    synthetic_target_keys = set(
        synthetic_targets[
            "_key"
        ]
    )

    instances_by_key: Dict[
        Tuple[str, str],
        Any,
    ] = {}

    instances_by_key.update(
        load_combined_instances(
            combined_target_keys
        )
    )

    instances_by_key.update(
        load_synthetic_instances(
            synthetic_target_keys
        )
    )

    expected_total = (
        EXPECTED_COMBINED_N
        + EXPECTED_SYNTHETIC_N
    )

    if len(
        instances_by_key
    ) != expected_total:
        raise RuntimeError(
            "Unexpected total graph-instance coverage: "
            f"{len(instances_by_key)} vs expected {expected_total}"
        )

    print(
        f"[{_ts()}] Graph-instance audit PASSED: "
        f"{EXPECTED_COMBINED_N} Combined + "
        f"{EXPECTED_SYNTHETIC_N} Synthetic'.",
        flush=True,
    )

    # --------------------------------------------------------
    # ONLY AFTER ALL AUDITS: RESET OUTPUT
    # --------------------------------------------------------
    reset_output_root()

    split_manifest.to_csv(
        OUTPUT_ROOT
        / "combined_train_test_split_manifest.csv",
        index=False,
    )

    removed_combined.to_csv(
        OUTPUT_ROOT
        / "modeling_removed_combined.csv",
        index=False,
    )

    removed_synthetic.to_csv(
        OUTPUT_ROOT
        / "modeling_removed_synthetic.csv",
        index=False,
    )

    # --------------------------------------------------------
    # EXACT FOUR-RUN PLAN
    # --------------------------------------------------------
    run_plan = [
        (
            1,
            "combined_to_combined",
            "classification",
        ),
        (
            2,
            "combined_to_combined",
            "regression",
        ),
        (
            3,
            "synthetic_to_combined",
            "classification",
        ),
        (
            4,
            "synthetic_to_combined",
            "regression",
        ),
    ]

    pd.DataFrame(
        [
            {
                "run_id": rid,
                "train_setting": train_setting,
                "task": task,
                "input": "static_tripartite_graph_only",
            }
            for (
                rid,
                train_setting,
                task,
            ) in run_plan
        ]
    ).to_csv(
        EXPERIMENT_MANIFEST_FILE,
        index=False,
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print(
        f"[{_ts()}] Device: {device}",
        flush=True,
    )

    results = []

    for (
        run_id,
        train_setting,
        task,
    ) in run_plan:
        row = run_one_experiment(
            run_id=run_id,
            train_setting=train_setting,
            task=task,
            combined_targets=combined_targets,
            synthetic_targets=synthetic_targets,
            combined_train_keys=combined_train_keys,
            combined_test_keys=combined_test_keys,
            instances_by_key=instances_by_key,
            device=device,
        )

        results.append(
            row
        )

        pd.DataFrame(
            results
        ).to_csv(
            MASTER_RESULTS_FILE,
            index=False,
        )

    results_df = pd.DataFrame(
        results
    )

    results_df.to_csv(
        MASTER_RESULTS_FILE,
        index=False,
    )

    table_rows = []

    for train_test in [
        "Combined→Combined",
        "Synthetic′→Combined",
    ]:
        sub = results_df[
            results_df[
                "train_test"
            ] == train_test
        ]

        clf = sub[
            sub[
                "task"
            ] == "classification"
        ]

        reg = sub[
            sub[
                "task"
            ] == "regression"
        ]

        table_rows.append(
            {
                "train_test": train_test,
                "GNN_C_accuracy": (
                    float(
                        clf.iloc[0][
                            "accuracy"
                        ]
                    )
                    if len(clf)
                    else np.nan
                ),
                "GNN_C_balanced_accuracy": (
                    float(
                        clf.iloc[0][
                            "balanced_accuracy"
                        ]
                    )
                    if len(clf)
                    else np.nan
                ),
                "GNN_C_f1_macro": (
                    float(
                        clf.iloc[0][
                            "f1_macro"
                        ]
                    )
                    if len(clf)
                    else np.nan
                ),
                "GNN_R_rAccuracy_paper": (
                    float(
                        reg.iloc[0][
                            "r_accuracy_paper"
                        ]
                    )
                    if len(reg)
                    else np.nan
                ),
                "GNN_R_rAccuracy_combined": (
                    float(
                        reg.iloc[0][
                            "r_accuracy_combined"
                        ]
                    )
                    if len(reg)
                    else np.nan
                ),
                "GNN_R_RMSE": (
                    float(
                        reg.iloc[0][
                            "rmse"
                        ]
                    )
                    if len(reg)
                    else np.nan
                ),
            }
        )

    thesis_table = pd.DataFrame(
        table_rows
    )

    thesis_table.to_csv(
        THESIS_TABLE_FILE,
        index=False,
    )

    print(
        "\n"
        + "=" * 90
    )

    print(
        f"[{_ts()}] ALL 4 STATIC-GNN EXPERIMENTS COMPLETE"
    )

    print(
        "=" * 90
    )

    print(
        thesis_table.to_string(
            index=False
        )
    )

    print(
        f"\n[{_ts()}] Master results -> "
        f"{MASTER_RESULTS_FILE}"
    )

    print(
        f"[{_ts()}] Thesis table   -> "
        f"{THESIS_TABLE_FILE}"
    )

    print(
        f"[{_ts()}] Output root    -> "
        f"{OUTPUT_ROOT}"
    )


if __name__ == "__main__":
    main()



KeyboardInterrupt

